In [1]:
# MATRIZ DE COVARIANZAS USANDO LOS PRECIOS DE CIERRE

"""
var_cov_tool.py — versión que usa exclusivamente el campo "close"

Calcula la matriz de covarianzas (y opcionalmente el VaR a 180 días en USD)
a partir de datos diarios de contratos de futuros (estructura { "ZCZ25": [ {date, close, ...}, ... ] }).
"""

from __future__ import annotations
from pathlib import Path
from typing import List, Dict, Union
import json
import pandas as pd
import numpy as np

# ======================== CONFIGURACIÓN ========================
json_paths = [
    r"C:\Users\DanielAristizabal\Saman\Banca de Inversión - Documents\Super de Alimentos\Gestión de riesgo\Data\data_sb.json",
    r"C:\Users\DanielAristizabal\Saman\Banca de Inversión - Documents\Super de Alimentos\Gestión de riesgo\Data\data_zc1.json",
    r"C:\Users\DanielAristizabal\Saman\Banca de Inversión - Documents\Super de Alimentos\Gestión de riesgo\Data\data_cc.json",
]

# Contratos exactos a analizar (tal como están en las claves del JSON)
# MAÍZ (ZC): H=Mar, K=May, N=Jul, U=Sep, Z=Dec
# AZÚCAR (SB): H=Mar, K=May, N=Jul, V=Oct (nota: usa 1 dígito para año)
# CACAO (CC): H=Mar, K=May, N=Jul, U=Sep, Z=Dec

selected_contracts = [
    # MAÍZ 2026
    "ZCH26", "ZCK26", "ZCN26", "ZCU26", "ZCZ26",
    # MAÍZ 2027
    "ZCH27", "ZCK27", "ZCN27", "ZCU27", "ZCZ27",
    # AZÚCAR 2026 (formato SBx6)
    "SBH6", "SBK6", "SBN6", "SBV6",
    # AZÚCAR 2027 (formato SBx7)
    "SBH7", "SBK7", "SBN7", "SBV7",
    # CACAO 2026
    "CCH26", "CCK26", "CCN26", "CCU26", "CCZ26",
    # CACAO 2027
    "CCH27", "CCK27", "CCN27", "CCU27", "CCZ27",
]

# Observaciones por contrato
n_obs = 180

# Usar Ledoit–Wolf (requiere scikit-learn)
use_shrinkage = True

# (Opcional) exposiciones en USD
exposures_usd = None

# Ruta de salida Excel
OUTPUT_EXCEL = Path(r"C:\Users\DanielAristizabal\Saman\Banca de Inversión - Documents\Super de Alimentos\Gestión de riesgo\Resultados\var_cov_output.xlsx")
# ===============================================================

# ---------------------------------------------------------------
# FUNCIONES
# ---------------------------------------------------------------

def _file_exists_or_raise(path: Union[str, Path]) -> None:
    if not Path(path).exists():
        raise FileNotFoundError(f"No existe el archivo: {path}")

def _read_json(path: Union[str, Path]) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def load_jsons_to_prices(json_paths: List[Union[str, Path]]) -> pd.DataFrame:
    all_rows = []
    for path in json_paths:
        _file_exists_or_raise(path)
        obj = _read_json(path)
        if not isinstance(obj, dict):
            raise ValueError(f"Estructura inesperada en {path}: se esperaba un dict con claves de contratos.")
        for contract_key, rec_list in obj.items():
            if not isinstance(rec_list, list):
                continue
            for r in rec_list:
                try:
                    date = pd.to_datetime(r["date"])
                    price = float(r["close"])
                    all_rows.append({"date": date, "contract": contract_key, "price": price})
                except Exception:
                    continue

    if not all_rows:
        raise ValueError("No se obtuvieron registros válidos de los JSON.")

    df = pd.DataFrame(all_rows)
    df.sort_values(["contract", "date"], inplace=True)
    price_df = (df.pivot_table(index="date", columns="contract", values="price", aggfunc="last").sort_index())
    return price_df

def last_n_trade_obs(series: pd.Series, n: int = 180) -> pd.Series:
    s = series.dropna()
    return s.iloc[-n:] if len(s) > 0 else s

def log_returns(prices: pd.Series) -> pd.Series:
    return np.log(prices / prices.shift(1)).dropna()

def build_returns(price_df: pd.DataFrame, selected_contracts: List[str], n_obs: int = 180) -> pd.DataFrame:
    missing = [c for c in selected_contracts if c not in price_df.columns]
    if missing:
        print(f"⚠️ Contratos no encontrados (se omiten): {missing}")
        selected_contracts = [c for c in selected_contracts if c in price_df.columns]
    if not selected_contracts:
        raise ValueError("No hay contratos válidos para calcular covarianza.")
    rets = {}
    for col in selected_contracts:
        px_n = last_n_trade_obs(price_df[col], n=n_obs)
        rets[col] = log_returns(px_n)
    R = pd.concat(rets.values(), axis=1, keys=rets.keys()).dropna()
    return R

def shrink_cov_ledoitwolf(R: pd.DataFrame) -> pd.DataFrame:
    try:
        from sklearn.covariance import LedoitWolf
        lw = LedoitWolf().fit(R.values)
        return pd.DataFrame(lw.covariance_, index=R.columns, columns=R.columns)
    except Exception:
        return R.cov()

def compute_covariance(R: pd.DataFrame, use_shrinkage: bool = True) -> pd.DataFrame:
    return shrink_cov_ledoitwolf(R) if use_shrinkage else R.cov()

def var_usd_from_cov(cov: pd.DataFrame, exposures_usd: Dict[str, float], alpha: float = 0.95, horizon_days: int = 180) -> float:
    zmap = {0.90: 1.2816, 0.95: 1.6449, 0.975: 1.9600, 0.99: 2.3263}
    z = zmap.get(alpha, 1.6449)
    e = pd.Series(exposures_usd, dtype=float).reindex(cov.columns)
    if e.isna().any():
        raise ValueError(f"Faltan exposiciones para: {list(e.index[e.isna()])}")
    e_vec = e.values.reshape(-1, 1)
    var_day_usd = float(e_vec.T @ cov.values @ e_vec)
    std_day_usd = np.sqrt(max(var_day_usd, 0.0))
    return float(z * std_day_usd * np.sqrt(horizon_days))

# ---------------------------------------------------------------
# PIPELINE PRINCIPAL
# ---------------------------------------------------------------

def run_pipeline():
    price_df = load_jsons_to_prices(json_paths)
    
    # Mostrar contratos disponibles
    print(f"Contratos disponibles en JSON: {list(price_df.columns)}\n")
    
    R = build_returns(price_df, selected_contracts, n_obs)
    cov = compute_covariance(R, use_shrinkage)
    
    # Ordenar por tipo de contrato
    cols_order = [c for c in selected_contracts if c in cov.columns]
    cov = cov.loc[cols_order, cols_order]

    print("=== Matriz de Covarianzas (rendimientos diarios, usando CLOSE) ===")
    print(f"Contratos incluidos: {list(cov.columns)}")
    print(cov.round(8))

    if exposures_usd:
        var_usd = var_usd_from_cov(cov, exposures_usd, alpha=0.95, horizon_days=180)
        print(f"\n=== VaR 95% a 180 días (USD) ===\n{round(var_usd, 2)}")

    return cov, R

# ---------------------------------------------------------------
# EJECUCIÓN Y GUARDADO
# ---------------------------------------------------------------
cov, R = run_pipeline()

# Guardar en Excel
OUTPUT_EXCEL.parent.mkdir(parents=True, exist_ok=True)
cov.to_excel(OUTPUT_EXCEL, sheet_name='Covarianza')
print(f"\n✅ Guardado en: {OUTPUT_EXCEL}")

Contratos disponibles en JSON: ['CCH25', 'CCH26', 'CCH27', 'CCK25', 'CCK26', 'CCK27', 'CCN25', 'CCN26', 'CCN27', 'CCU25', 'CCU26', 'CCU27', 'CCZ25', 'CCZ26', 'CCZ27', 'SBH26', 'SBH27', 'SBH4', 'SBH5', 'SBH6', 'SBH7', 'SBH8', 'SBK26', 'SBK4', 'SBK5', 'SBK6', 'SBK7', 'SBK8', 'SBN26', 'SBN4', 'SBN5', 'SBN6', 'SBN7', 'SBN8', 'SBV26', 'SBV4', 'SBV5', 'SBV6', 'SBV7', 'ZCH25', 'ZCH26', 'ZCH27', 'ZCH28', 'ZCK25', 'ZCK26', 'ZCK27', 'ZCN25', 'ZCN26', 'ZCN27', 'ZCN28', 'ZCN29', 'ZCU25', 'ZCU26', 'ZCU27', 'ZCZ25', 'ZCZ26', 'ZCZ27', 'ZCZ28', 'ZCZ29']

=== Matriz de Covarianzas (rendimientos diarios, usando CLOSE) ===
Contratos incluidos: ['ZCH26', 'ZCK26', 'ZCN26', 'ZCU26', 'ZCZ26', 'ZCH27', 'ZCK27', 'ZCN27', 'ZCU27', 'ZCZ27', 'SBH6', 'SBK6', 'SBN6', 'SBV6', 'SBH7', 'SBK7', 'SBN7', 'SBV7', 'CCH26', 'CCK26', 'CCN26', 'CCU26', 'CCZ26', 'CCH27', 'CCK27', 'CCN27', 'CCU27', 'CCZ27']
              ZCH26     ZCK26     ZCN26     ZCU26     ZCZ26     ZCH27  \
ZCH26  1.467800e-04  0.000113  0.000106  0.000075

In [1]:
# COVARIANZA / VARIANZA PARA USDCOP (TRM) 
#
# - Lee TRM desde Excel: 1ra columna = fecha, 2da columna = precio
# - Calcula retornos logarítmicos diarios
# - Calcula varianza (covarianza 1x1). Opcional Ledoit–Wolf (no aporta mucho en 1 dimensión)
# - (Opcional) VaR 95% a 180 días en USD si defines exposure en USD
#
# Ejecuta: python cov_usdcop_only.py

from __future__ import annotations
from pathlib import Path
from typing import Union, Optional
import pandas as pd
import numpy as np

# ======================== CONFIGURACIÓN ========================
trm_excel_path = Path(r"C:\Users\DanielAristizabal\Saman\Banca de Inversión - Documents\Super de Alimentos\Gestión de riesgo\Data\trm.xlsx")
trm_sheet_name = "sheet1"

label = "USDCOP"
n_obs = 180

# Ledoit–Wolf (requiere scikit-learn). En 1D, var = varianza muestral; shrinkage es opcional.
use_shrinkage = False

# (Opcional) exposición en USD para VaR (ej: si tienes un flujo/posición en USD que se convierte a COP)
exposure_usd: Optional[float] = None  # ej: 1_000_000

alpha = 0.95
horizon_days = 180
# ===============================================================


def _file_exists_or_raise(path: Union[str, Path]) -> None:
    if not Path(path).exists():
        raise FileNotFoundError(f"No existe el archivo: {path}")

def load_trm_from_excel_two_cols(excel_path: Union[str, Path], sheet_name: str, label: str = "USDCOP") -> pd.Series:
    """
    Lee TRM desde Excel asumiendo:
      - 1ra columna = fecha
      - 2da columna = precio (USDCOP)
    """
    _file_exists_or_raise(excel_path)
    df = pd.read_excel(excel_path, sheet_name=sheet_name)

    if df.shape[1] < 2:
        raise ValueError("TRM: el Excel debe tener al menos 2 columnas (fecha y precio).")

    df2 = df.iloc[:, :2].copy()
    df2.columns = ["fecha", "precio"]

    df2["fecha"] = pd.to_datetime(df2["fecha"], errors="coerce")

    # Soporta coma decimal si viniera como texto
    if df2["precio"].dtype == object:
        df2["precio"] = (df2["precio"].astype(str)
                         .str.replace(".", "", regex=False)
                         .str.replace(",", ".", regex=False))

    df2["precio"] = pd.to_numeric(df2["precio"], errors="coerce")

    df2 = (df2.dropna(subset=["fecha", "precio"])
               .sort_values("fecha")
               .drop_duplicates(subset=["fecha"], keep="last"))

    s = df2.set_index("fecha")["precio"]
    s.name = label
    return s

def log_returns(prices: pd.Series) -> pd.Series:
    return np.log(prices / prices.shift(1)).dropna()

def shrink_var_ledoitwolf(r: pd.Series) -> float:
    """
    Shrinkage para varianza 1D. En práctica, la varianza muestral suele ser suficiente.
    """
    try:
        from sklearn.covariance import LedoitWolf
        lw = LedoitWolf().fit(r.values.reshape(-1, 1))
        return float(lw.covariance_[0, 0])
    except Exception:
        return float(r.var(ddof=1))

def compute_var(r: pd.Series, use_shrinkage: bool = False) -> float:
    return shrink_var_ledoitwolf(r) if use_shrinkage else float(r.var(ddof=1))

def var_usd_from_var(var_daily: float, exposure_usd: float, alpha: float = 0.95, horizon_days: int = 180) -> float:
    zmap = {0.90: 1.2816, 0.95: 1.6449, 0.975: 1.9600, 0.99: 2.3263}
    z = zmap.get(alpha, 1.6449)
    std_daily = np.sqrt(max(var_daily, 0.0))
    # Si exposure_usd es notional en USD, la desviación del PnL en USD ~ exposure_usd * std_return
    return float(z * abs(exposure_usd) * std_daily * np.sqrt(horizon_days))

def run_pipeline():
    s = load_trm_from_excel_two_cols(trm_excel_path, trm_sheet_name, label=label)

    # Tomar últimas n_obs+1 para obtener n_obs retornos
    s_win = s.dropna().iloc[-(n_obs + 1):]
    r = log_returns(s_win)

    if len(r) < 10:
        raise ValueError(f"Muy pocos retornos para estimar varianza: {len(r)}. Revisa el Excel/hoja/rango.")

    var_daily = compute_var(r, use_shrinkage=use_shrinkage)

    cov = pd.DataFrame([[var_daily]], index=[label], columns=[label])

    print("=== USDCOP (TRM) — Varianza/Covarianza 1x1 (retornos log) ===")
    print(cov.round(10))

    print("\n=== Diagnóstico ===")
    print(f"Precios: {s_win.index.min().date()} → {s_win.index.max().date()} | Obs precios: {len(s_win)}")
    print(f"Retornos: {r.index.min().date()} → {r.index.max().date()} | Obs retornos: {len(r)}")
    print(f"Std diaria (retornos): {np.sqrt(var_daily):.6f}")

    if exposure_usd is not None:
        var_180 = var_usd_from_var(var_daily, exposure_usd, alpha=alpha, horizon_days=horizon_days)
        print(f"\n=== VaR {int(alpha*100)}% a {horizon_days} días (USD) ===")
        print(round(var_180, 2))

    return cov, r, s_win

if __name__ == "__main__":
    run_pipeline()


=== USDCOP (TRM) — Varianza/Covarianza 1x1 (retornos log) ===
          USDCOP
USDCOP  0.000048

=== Diagnóstico ===
Precios: 2025-05-24 → 2026-03-03 | Obs precios: 181
Retornos: 2025-05-28 → 2026-03-03 | Obs retornos: 180
Std diaria (retornos): 0.006944


In [ ]:
# CORRELACIÓN: ACERO (HRC), DÓLAR (USDCOP), TASA DE INTERÉS (IBR)
import pandas as pd
import numpy as np

# ======================== CONFIGURACIÓN ========================
TRM_EXCEL = r"C:\Users\DanielAristizabal\Saman\Banca de Inversión - Documents\Super de Alimentos\Gestión de riesgo\Data\trm.xlsx"
TRM_SHEET = "sheet1"
HRC_CSV = r"Datos históricos Bobina de acero de EE.UU..csv"
IBR_EXCEL = r"C:\Users\DanielAristizabal\Saman\Banca de Inversión - Documents\Super de Alimentos\Gestión de riesgo\Data\ibr.xlsx"
IBR_SHEET = "sheet1"
# ===============================================================

# --- Cargar TRM (USDCOP) ---
df_trm = pd.read_excel(TRM_EXCEL, sheet_name=TRM_SHEET)
df_trm = df_trm.iloc[:, :2].copy()
df_trm.columns = ["fecha", "precio"]
df_trm["fecha"] = pd.to_datetime(df_trm["fecha"], errors="coerce")
if df_trm["precio"].dtype == object:
    df_trm["precio"] = (df_trm["precio"].astype(str)
                         .str.replace(".", "", regex=False)
                         .str.replace(",", ".", regex=False))
df_trm["precio"] = pd.to_numeric(df_trm["precio"], errors="coerce")
df_trm = df_trm.dropna().sort_values("fecha").drop_duplicates("fecha", keep="last")
trm = df_trm.set_index("fecha")["precio"]
trm.name = "USDCOP"

# --- Cargar Acero HRC ---
df_hrc = pd.read_csv(HRC_CSV)
df_hrc["close"] = df_hrc["Último"].str.replace(".", "", regex=False).str.replace(",", ".", regex=False).astype(float)
df_hrc["fecha"] = pd.to_datetime(df_hrc["Fecha"], format="%d.%m.%Y")
df_hrc = df_hrc.sort_values("fecha").drop_duplicates("fecha", keep="last")
hrc = df_hrc.set_index("fecha")["close"]
hrc.name = "HRC"

# --- Cargar IBR ---
df_ibr = pd.read_excel(IBR_EXCEL, sheet_name=IBR_SHEET)
df_ibr = df_ibr.iloc[:, :2].copy()
df_ibr.columns = ["fecha", "tasa"]
df_ibr["fecha"] = pd.to_datetime(df_ibr["fecha"], errors="coerce")
if df_ibr["tasa"].dtype == object:
    df_ibr["tasa"] = (df_ibr["tasa"].astype(str)
                       .str.replace(",", ".", regex=False))
df_ibr["tasa"] = pd.to_numeric(df_ibr["tasa"], errors="coerce")
df_ibr = df_ibr.dropna().sort_values("fecha").drop_duplicates("fecha", keep="last")
ibr = df_ibr.set_index("fecha")["tasa"]
ibr.name = "IBR"

# --- Alinear por fechas comunes ---
precios = pd.concat([hrc, trm, ibr], axis=1).dropna()
print(f"Fechas comunes: {len(precios)}")
print(f"Rango: {precios.index.min().date()} a {precios.index.max().date()}\n")

# --- Variaciones diarias ---
# HRC y USDCOP: retornos logarítmicos (son precios)
# IBR: cambio absoluto (es una tasa de interés, no un precio)
ret_hrc = np.log(precios["HRC"] / precios["HRC"].shift(1))
ret_trm = np.log(precios["USDCOP"] / precios["USDCOP"].shift(1))
ret_ibr = precios["IBR"] - precios["IBR"].shift(1)  # delta en pp

retornos = pd.concat([ret_hrc, ret_trm, ret_ibr], axis=1).dropna()

# --- Matriz de correlación ---
corr = retornos.corr()

print("=" * 55)
print("MATRIZ DE CORRELACIÓN")
print("HRC y USDCOP: retornos log | IBR: cambios absolutos")
print("=" * 55)
display(corr.round(4))

# --- Resumen por pares ---
print(f"\n{'─' * 55}")
pares = [("HRC", "USDCOP"), ("HRC", "IBR"), ("USDCOP", "IBR")]
for a, b in pares:
    rho = corr.loc[a, b]
    fuerza = "fuerte" if abs(rho) > 0.7 else "moderada" if abs(rho) > 0.4 else "débil"
    signo = "Positiva" if rho > 0 else "Negativa"
    print(f"  {a:8s} vs {b:8s} → ρ = {rho:+.4f}  ({signo} {fuerza})")

print(f"\nObservaciones utilizadas: {len(retornos)}")